# drfn: surface-form routing probe (CPU, no GPU quota)

Reads the finished checkpoints of `drfn-grounded-runs` and `drfn-grounded-entropy` and measures
routing consistency the way section 4.6 defines it: one vote per **surface form**, not per occurrence.
The token-level number printed at the end of each training run is inflated by repeated frequent words.

All runs are probed against the real geezRPA labels, including the placebo runs.
Also reported: the same score restricted to dev forms **unseen in training** together with their seen
siblings, and for grounded runs, how often an unseen form routes to its own root's slot.

In [ ]:
!pip install -q tokenizers


In [ ]:
open('/kaggle/working/grounded_new.py', 'w').write('"""Grounded Morphological Memory (GMM).\n\nA drop-in replacement for cmm.CompositionalMorphMemory that changes two\nstructural things and fixes three places where the implemented CMM departs\nfrom the written protocol (mmm_positioning.pdf, Eqs. 4-12).\n\nWHAT IS NEW (the architectural contribution)\n--------------------------------------------\n1. Routing grounding.  The root/pattern labels supervise the memory\'s OWN\n   routing distributions A^r, A^a -- not an external head whose output is\n   discarded.  Slot k of M_V^r therefore *is* root class k.  The retrieved\n   vector R^r = A^r M_V^r is on the computational path to the decoder, so\n   the supervision installs persistent structure in weights that survive to\n   inference, instead of only shaping the encoder state by gradient\n   pressure.  Labels remain training-only and absent at inference (C3).\n\n2. Word-level routing.  A Semitic root is a property of a word, not of a\n   BPE piece.  Routing once per word (mean-pool the word\'s subword span,\n   route, broadcast the retrieval back) makes the assignment well-posed.\n   The current code instead routes per subword and patches the mismatch on\n   the label side by copying a word\'s label onto every piece.\n\nBoth are orthogonal, giving the 2x2 ablation\n    {subword, word} routing  x  {free, grounded}\nin which the published CMM is the (subword, free) cell.\n\nWHAT IS RESTORED (protocol fidelity)\n------------------------------------\na. Untied keys and values.  cmm.py uses one `root_bank` as both key and\n   value with no query projection:  softmax(h B^T) B.  The protocol Eqs.\n   (4)-(5) specify Q = H W_Q and separate M_K (K x d_k), M_V (K x d).  With\n   tied K/V the retrieved vector is pulled toward whichever slot is already\n   closest to h in embedding space, which biases slots toward surface\n   similarity rather than abstract roots.\nb. Scalar gate.  Protocol Eq. (9) is a per-token scalar\n   lambda_t = sigma(w^T [h_t ; r_t] + b).  cmm.py uses a d-dimensional\n   elementwise gate, which makes the §4.6 "gate statistics by token class"\n   probe ill-defined.  `scalar_gate=True` restores Eq. (9).\nc. Explicit entropy sign.  §3.6 says "Minimizing L_ent sharpens each\n   token\'s routing", and Eq. (12) adds +alpha * L_ent.  train.py computes\n   `total = total - entropy_reg * ent`, i.e. it MAXIMIZES entropy and\n   drives routing toward uniform -- the paper\'s own failure mode (i).\n   `entropy_sign` makes the choice explicit and named; see grounding_loss.\n\nSet `grounded=False, word_level=False, scalar_gate=False, tied_kv=True`\nto recover the published CMM exactly.\n"""\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nEPS = 1e-9\n\n\ndef pool_words(h, word_ids, n_words):\n    """Mean-pool token states into word slots.\n\n    h         : (B, T, d)\n    word_ids  : (B, T) long, word index within the sentence, -1 where the\n                position is not part of a word (BOS/EOS/PAD).\n    n_words   : int, max words per sentence in this batch (W).\n\n    Returns (pooled, flat_index, valid) where pooled is (B*W, d).\n    """\n    b, t, d = h.shape\n    valid = word_ids >= 0\n    offs = torch.arange(b, device=h.device).unsqueeze(1) * n_words\n    flat_index = (word_ids.clamp_min(0) + offs).reshape(-1)\n    hf = h.reshape(b * t, d)\n    vf = valid.reshape(-1)\n\n    pooled = h.new_zeros(b * n_words, d)\n    counts = h.new_zeros(b * n_words)\n    pooled.index_add_(0, flat_index[vf], hf[vf])\n    counts.index_add_(0, flat_index[vf], torch.ones_like(counts[:1]).expand(int(vf.sum())))\n    pooled = pooled / counts.clamp_min(1.0).unsqueeze(-1)\n    return pooled, flat_index, valid\n\n\nclass GroundedMorphMemory(nn.Module):\n    """Factorized persistent memory with optionally grounded routing.\n\n    n_root_slots / n_pattern_slots: when `grounded=True` these MUST equal\n    the label inventory sizes, because slot k is identified with class k.\n    """\n\n    def __init__(self, d_model, n_root_slots=256, n_pattern_slots=17,\n                 d_key=None, with_binding=True, word_level=True,\n                 grounded=True, scalar_gate=True, tied_kv=False,\n                 dropout=0.1):\n        super().__init__()\n        self.d_model = d_model\n        self.d_key = d_key or max(1, d_model // 4)\n        self.n_root_slots = n_root_slots\n        self.n_pattern_slots = n_pattern_slots\n        self.with_binding = with_binding\n        self.word_level = word_level\n        self.grounded = grounded\n        self.scalar_gate = scalar_gate\n        self.tied_kv = tied_kv\n\n        if tied_kv:\n            self.root_bank = nn.Parameter(\n                torch.randn(n_root_slots, d_model) / math.sqrt(d_model))\n            self.pattern_bank = nn.Parameter(\n                torch.randn(n_pattern_slots, d_model) / math.sqrt(d_model))\n        else:\n            self.wq_root = nn.Linear(d_model, self.d_key)\n            self.wq_pat = nn.Linear(d_model, self.d_key)\n            self.mk_root = nn.Parameter(torch.empty(n_root_slots, self.d_key))\n            self.mv_root = nn.Parameter(torch.empty(n_root_slots, d_model))\n            self.mk_pat = nn.Parameter(torch.empty(n_pattern_slots, self.d_key))\n            self.mv_pat = nn.Parameter(torch.empty(n_pattern_slots, d_model))\n            for p in (self.mk_root, self.mv_root, self.mk_pat, self.mv_pat):\n                nn.init.xavier_uniform_(p)\n\n        comp_dim = 3 * d_model if with_binding else 2 * d_model\n        self.compose = nn.Linear(comp_dim, d_model)\n        if scalar_gate:\n            self.gate = nn.Linear(2 * d_model, 1)          # Eq. (9)\n        else:\n            self.gate_h = nn.Linear(d_model, d_model)\n            self.gate_c = nn.Linear(d_model, d_model)\n        self.dropout = nn.Dropout(dropout)\n        self.norm = nn.LayerNorm(d_model)\n\n    def _route(self, x, which):\n        """x: (N, d) -> (scores (N, K), retrieved (N, d))."""\n        if self.tied_kv:\n            bank = self.root_bank if which == "root" else self.pattern_bank\n            logits = x @ bank.t() / math.sqrt(self.d_model)\n            scores = F.softmax(logits, dim=-1)\n            return scores, scores @ bank\n        wq = self.wq_root if which == "root" else self.wq_pat\n        mk = self.mk_root if which == "root" else self.mk_pat\n        mv = self.mv_root if which == "root" else self.mv_pat\n        logits = wq(x) @ mk.t() / math.sqrt(self.d_key)\n        scores = F.softmax(logits, dim=-1)\n        return scores, scores @ mv\n\n    def forward(self, h, key_padding_mask=None, word_ids=None):\n        b, t, d = h.shape\n\n        if self.word_level:\n            if word_ids is None:\n                raise ValueError("word_level=True requires word_ids")\n            n_words = max(int(word_ids.max().item()) + 1, 1)\n            x, flat_index, valid = pool_words(h, word_ids, n_words)\n        else:\n            x = h.reshape(b * t, d)\n\n        root_scores, r = self._route(x, "root")            # Eqs. (4)-(5)\n        pat_scores, a = self._route(x + r, "pattern")      # Eqs. (6)-(7)\n\n        comp = torch.cat([r, a, r * a], -1) if self.with_binding \\\n            else torch.cat([r, a], -1)\n        c = self.compose(comp)                             # Eq. (8)\n\n        if self.word_level:\n            # broadcast word-level retrievals and routings back to subwords\n            c = c[flat_index].reshape(b, t, d) * valid.unsqueeze(-1)\n            root_scores_tok = root_scores[flat_index].reshape(b, t, -1)\n            pat_scores_tok = pat_scores[flat_index].reshape(b, t, -1)\n        else:\n            c = c.reshape(b, t, d)\n            root_scores_tok = root_scores.reshape(b, t, -1)\n            pat_scores_tok = pat_scores.reshape(b, t, -1)\n\n        if self.scalar_gate:\n            lam = torch.sigmoid(self.gate(torch.cat([h, c], -1)))   # (b,t,1)\n        else:\n            lam = torch.sigmoid(self.gate_h(h) + self.gate_c(c))\n        out = self.norm(h + self.dropout(lam * c))                  # Eq. (9)\n        if key_padding_mask is not None:\n            out = out.masked_fill(key_padding_mask.unsqueeze(-1), 0.0)\n\n        info = {"root_scores": root_scores_tok,\n                "pattern_scores": pat_scores_tok,\n                "gate": lam.detach(),\n                "grounded": self.grounded}\n        return out, info\n\n\ndef grounding_loss(routings, root_lab, pat_lab, root_weight=0.5,\n                   pattern_weight=0.5, entropy_weight=0.0,\n                   entropy_sign=+1.0, balance_weight=0.0,\n                   ignore_index=-100, ignore_pattern_ids=()):\n    """Cross-entropy on the memory\'s own routing distributions.\n\n    routings : list of info dicts returned by GroundedMorphMemory.forward\n    root_lab, pat_lab : (B, T) long, ignore_index where unlabelled.\n\n    entropy_sign = +1 minimizes routing entropy (sharpening, as §3.6\n    specifies); -1 reproduces train.py\'s current anti-collapse bonus, which\n    maximizes entropy.  Pass the sign explicitly so the choice is recorded\n    in the run config rather than implied by a minus in the trainer.\n\n    Returns (total, terms).\n    """\n    terms = {}\n    total = None\n    if not routings:\n        return None, terms\n\n    if ignore_pattern_ids:\n        pat_lab = pat_lab.clone()\n        for pid in ignore_pattern_ids:\n            pat_lab[pat_lab == pid] = ignore_index\n\n    ce_r, ce_p, ents, bals = [], [], [], []\n    for info in routings:\n        if not info.get("grounded"):\n            continue\n        rs = info["root_scores"].clamp_min(EPS)\n        ps = info["pattern_scores"].clamp_min(EPS)\n        ce_r.append(F.nll_loss(rs.log().reshape(-1, rs.size(-1)),\n                               root_lab.reshape(-1),\n                               ignore_index=ignore_index))\n        ce_p.append(F.nll_loss(ps.log().reshape(-1, ps.size(-1)),\n                               pat_lab.reshape(-1),\n                               ignore_index=ignore_index))\n        for s in (rs, ps):\n            ents.append(-(s * s.log()).sum(-1).mean())\n            mean_assign = s.mean(dim=(0, 1))\n            uni = torch.full_like(mean_assign, 1.0 / mean_assign.size(-1))\n            bals.append((mean_assign * (mean_assign.log() - uni.log())).sum())\n\n    if not ce_r:\n        return None, terms\n    l_root = torch.stack(ce_r).mean()\n    l_pat = torch.stack(ce_p).mean()\n    total = root_weight * l_root + pattern_weight * l_pat\n    terms["route_root"] = l_root.detach()\n    terms["route_pattern"] = l_pat.detach()\n    if entropy_weight:\n        ent = torch.stack(ents).mean()\n        total = total + entropy_sign * entropy_weight * ent\n        terms["entropy"] = ent.detach()\n    if balance_weight:\n        bal = torch.stack(bals).mean()\n        total = total + balance_weight * bal\n        terms["balance"] = bal.detach()\n    return total, terms\n\n\n@torch.no_grad()\ndef routing_consistency_types(model, pairs, tokenizer, label_entries, cfg,\n                              device, layer=0, train_types=None,\n                              min_types=2, n_permutations=200, seed=0):\n    """Section 4.6 probe 1, measured over SURFACE FORMS, not tokens.\n\n    The token-level `routing_consistency` below is inflated by repetition:\n    every occurrence of a frequent word routes the same way, so a root whose\n    tokens are mostly one surface form scores high with no abstraction at all,\n    and the token-shuffling baseline does not control for it. Section 4.6\n    defines the probe over "attested surface forms", which this implements:\n\n      1. word slot  = argmax of the routing distribution averaged over the\n                      word\'s BPE pieces (identical pieces for word-level\n                      routing; a real average for subword routing);\n      2. type slot  = majority slot over that word type\'s occurrences;\n      3. root score = fraction of the root\'s types sharing its majority slot,\n                      over roots with >= min_types distinct types;\n      4. baseline   = same score with root labels permuted ACROSS TYPES,\n                      preserving the type-level root marginal.\n\n    If `train_types` is given, also reports the probe restricted to types\n    UNSEEN in training whose root has a seen sibling, which is the\n    mechanism-level version of the held-out paradigm test. For grounded\n    models (slot k = class k) it additionally reports slot accuracy on those\n    unseen types: does a new inflection route to its root\'s own slot?\n\n    Returns a dict; values are None when there is not enough data.\n    """\n    import collections\n    import random as _rnd\n    from data import encode_words, word_ids_from_starts\n\n    model.eval()\n    occ = collections.defaultdict(list)          # word type -> [slot, ...]\n    for src, tgt in pairs:\n        ids, starts = encode_words(tokenizer, src, cfg["bos_id"],\n                                   cfg["eos_id"], cfg["max_len"])\n        n = len(ids)\n        wid = word_ids_from_starts(starts, n)\n        s = torch.tensor([ids], device=device)\n        t_ = torch.tensor([[cfg["bos_id"], cfg["eos_id"]]], device=device)\n        out = model(s, t_, wid.unsqueeze(0).to(device))\n        rs = [r for r in out.get("routings", []) if "root_scores" in r]\n        if not rs:\n            return {"error": "model has no routing memory"}\n        A = rs[layer]["root_scores"][0]                   # (n, K)\n        words = src.split()\n        for k in range(len(starts)):\n            m = (wid == k)\n            if not m.any():\n                continue\n            occ[words[k]].append(int(A[m.to(A.device)].mean(0).argmax()))\n\n    type_slot = {w: collections.Counter(v).most_common(1)[0][0]\n                 for w, v in occ.items()}\n    type_root = {w: int(label_entries.get(w, {}).get("root", -100))\n                 for w in type_slot}\n    type_root = {w: r for w, r in type_root.items() if r >= 1}\n\n    def score(assign, types):\n        by = collections.defaultdict(list)\n        for w in types:\n            by[assign[w]].append(type_slot[w])\n        vals = [collections.Counter(v).most_common(1)[0][1] / len(v)\n                for v in by.values() if len(v) >= min_types]\n        return (sum(vals) / len(vals), len(vals)) if vals else (None, 0)\n\n    def with_baseline(types):\n        types = sorted(types)\n        obs, n_roots = score(type_root, types)\n        if obs is None:\n            return {"consistency": None, "permutation": None, "n_roots": 0,\n                    "n_types": len(types)}\n        rng = _rnd.Random(seed)\n        labs = [type_root[w] for w in types]\n        perms = []\n        for _ in range(n_permutations):\n            rng.shuffle(labs)\n            v, _ = score(dict(zip(types, labs)), types)\n            if v is not None:\n                perms.append(v)\n        return {"consistency": obs,\n                "permutation": sum(perms) / len(perms) if perms else None,\n                "n_roots": n_roots, "n_types": len(types)}\n\n    res = {"all_types": with_baseline(type_root.keys())}\n\n    if train_types is not None:\n        seen_roots = {r for w, r in type_root.items() if w in train_types}\n        unseen = [w for w, r in type_root.items()\n                  if w not in train_types and r in seen_roots]\n        # consistency of unseen forms WITH their seen siblings\n        fam = [w for w, r in type_root.items()\n               if r in {type_root[u] for u in unseen}]\n        res["unseen_with_siblings"] = with_baseline(fam)\n        res["n_unseen_types"] = len(unseen)\n        if unseen:\n            acc = sum(type_slot[w] == type_root[w] for w in unseen) / len(unseen)\n            res["unseen_slot_accuracy"] = acc   # meaningful only if grounded\n        else:\n            res["unseen_slot_accuracy"] = None\n    return res\n\n\n@torch.no_grad()\ndef routing_consistency(model, loader, device, layer=0, level="root",\n                        min_forms=2, top_n=200, n_permutations=100, seed=0):\n    """§4.6 probe 1: do surface forms of one root share a routing slot?\n\n    For each label class with >= min_forms distinct surface positions,\n    computes the fraction of its tokens whose routing argmax equals the\n    class\'s majority slot.  The permutation baseline reshuffles the\n    class labels across tokens, preserving both marginals.\n\n    Returns {"consistency": float, "permutation": float, "n_classes": int}.\n    """\n    import collections\n    model.eval()\n    key = "root_scores" if level == "root" else "pattern_scores"\n    by_class = collections.defaultdict(list)\n\n    for src, tgt, root_lab, pat_lab, word_ids in loader:\n        src = src.to(device)\n        lab = (root_lab if level == "root" else pat_lab).to(device)\n        out = model(src, tgt[:, :-1].to(device), word_ids.to(device))\n        routings = [r for r in out.get("routings", []) if key in r]\n        if not routings:\n            return {"consistency": float("nan"), "permutation": float("nan"),\n                    "n_classes": 0}\n        arg = routings[layer][key].argmax(-1)\n        m = lab >= 1                      # skip <none> (0) and ignore (-100)\n        for cls, slot in zip(lab[m].tolist(), arg[m].tolist()):\n            by_class[cls].append(slot)\n\n    classes = sorted(by_class, key=lambda c: -len(by_class[c]))[:top_n]\n    classes = [c for c in classes if len(set(by_class[c])) >= 1\n               and len(by_class[c]) >= min_forms]\n    if not classes:\n        return {"consistency": float("nan"), "permutation": float("nan"),\n                "n_classes": 0}\n\n    def score(mapping):\n        vals = []\n        for c in classes:\n            slots = mapping[c]\n            maj = collections.Counter(slots).most_common(1)[0][1]\n            vals.append(maj / len(slots))\n        return sum(vals) / len(vals)\n\n    obs = score(by_class)\n\n    import random as _rnd\n    rng = _rnd.Random(seed)\n    pool = [s for c in classes for s in by_class[c]]\n    perms = []\n    for _ in range(n_permutations):\n        shuffled = pool[:]\n        rng.shuffle(shuffled)\n        mapping, i = {}, 0\n        for c in classes:\n            n = len(by_class[c])\n            mapping[c] = shuffled[i:i + n]\n            i += n\n        perms.append(score(mapping))\n    return {"consistency": obs, "permutation": sum(perms) / len(perms),\n            "n_classes": len(classes)}\n')
open('/kaggle/working/probe_runs.py', 'w').write('"""Apply the surface-form routing probe (section 4.6) to finished runs.\n\n    python probe_runs.py --pack PACK_DIR --label-cache LC.json \\\n        --runs RUN_DIR [RUN_DIR ...] --out probe_results.json\n\nEvery run is probed against the REAL label cache, including placebo runs:\nthe question is whether the memory groups surface forms of the same real\nroot, and a placebo-trained memory should not.\n"""\nimport argparse\nimport json\nimport os\nimport sys\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--pack", required=True, help="dir holding model.py etc.")\n    ap.add_argument("--label-cache", required=True)\n    ap.add_argument("--data-dir", default=None)\n    ap.add_argument("--runs", nargs="+", required=True)\n    ap.add_argument("--layer", type=int, default=0)\n    ap.add_argument("--limit", type=int, default=None)\n    ap.add_argument("--out", default="probe_results.json")\n    a = ap.parse_args()\n\n    sys.path.insert(0, a.pack)\n    import torch\n    import data as D\n    from model import ThesisTransformer\n    from grounded import routing_consistency_types\n\n    dd = a.data_dir or os.path.join(a.pack, "data")\n    train, _ = D.load_pairs(os.path.join(dd, "age_train.tsv"))\n    dev, _ = D.load_pairs(os.path.join(dd, "age_dev.tsv"))\n    if a.limit:\n        dev = dev[:a.limit]\n    train_types = {w for s, _ in train for w in s.split()}\n    entries = json.load(open(a.label_cache, encoding="utf-8"))["entries"]\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    results = {}\n    for rd in a.runs:\n        name = os.path.basename(os.path.normpath(rd))\n        ck_path = os.path.join(rd, "best.pt")\n        if not os.path.exists(ck_path):\n            results[name] = {"error": "no best.pt"}\n            print(f"{name}: no best.pt", flush=True)\n            continue\n        ck = torch.load(ck_path, map_location=device, weights_only=False)\n        cfg = ck["cfg"]\n        model = ThesisTransformer(cfg, cfg["n_root_classes"],\n                                  cfg["n_pattern_classes"])\n        model.load_state_dict(ck["model"])\n        model.to(device)\n        tok = D.load_tokenizer(os.path.join(rd, "tokenizer.json"))\n        r = routing_consistency_types(model, dev, tok, entries, cfg, device,\n                                      layer=a.layer, train_types=train_types)\n        r["variant"] = cfg.get("variant")\n        r["entropy_sign"] = cfg.get("entropy_sign")\n        r["grounded"] = bool(cfg.get("grounded_routing"))\n        results[name] = r\n        al = r.get("all_types", {})\n        un = r.get("unseen_with_siblings", {})\n        f = lambda v: "  n/a" if v is None else f"{v:.3f}"\n        print(f"{name:<24} all types {f(al.get(\'consistency\'))} vs perm "\n              f"{f(al.get(\'permutation\'))} ({al.get(\'n_roots\')} roots) | "\n              f"unseen+siblings {f(un.get(\'consistency\'))} vs perm "\n              f"{f(un.get(\'permutation\'))} | unseen slot acc "\n              f"{f(r.get(\'unseen_slot_accuracy\'))} (n={r.get(\'n_unseen_types\')})",\n              flush=True)\n    json.dump(results, open(a.out, "w"), indent=1)\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
import glob, os, shutil, subprocess, sys
WORK = '/kaggle/working/probe'
os.makedirs(WORK, exist_ok=True)
# model code + data come from the grid notebook's own output, so the probe
# uses exactly the code those checkpoints were trained with
pack_out = os.path.dirname(sorted(glob.glob('/kaggle/input/**/pack/model.py', recursive=True))[0])
for f in ('data.py', 'model.py', 'cmm.py'):
    shutil.copyfile(os.path.join(pack_out, f), os.path.join(WORK, f))
data_dir = os.path.join(pack_out, 'data')
lc = os.path.join(data_dir, 'geezrpa_label_cache_k256.json')
runs = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/runs/*/best.pt', recursive=True)})
print('code from', pack_out)
for r in runs: print('  run', r)
shutil.copyfile('/kaggle/working/grounded_new.py', os.path.join(WORK, 'grounded.py'))


In [ ]:
r = subprocess.run([sys.executable, '-u', '/kaggle/working/probe_runs.py', '--pack', WORK,
                    '--data-dir', data_dir, '--label-cache', lc, '--runs', *runs,
                    '--out', '/kaggle/working/probe_results.json'],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:])
